In [119]:
import pandas as pd

# define functions

def clean_ammonia_csv(csv_file_path, countries):
    
    # Read the CSV file into a DataFrame
    df = pd.read_csv(csv_file_path)
    # clean up the df by removing nan columns
    df = df.dropna(axis=1, how='all')
    # Filter the DataFrame by the specified country
    df_filtered = df[df['Country or Area'].isin(countries)]
    # Only keep volumes
    df_filtered = df_filtered[df_filtered['Unit'] == 't']
    # rename columns
    df_filtered.columns = ["country","element","year","unit", "value"]
    # Clean value column: remove commas and convert to float
    df_filtered['value'] = df_filtered['value'].replace({',': ''}, regex=True).astype(float)

    
    # Pivot the data to have Elements as columns, indexed by Country, Year, and Unit
    df_pivot = df_filtered.pivot_table(
        index=['country', 'year', 'unit'],
        columns='element',
        values='value',
        aggfunc='sum'
    )
    df_pivot.columns.name = None

    # Use .get() with fallback to avoid errors if a column is missing
    import_qty = pd.to_numeric(df_pivot['Import Quantity'] if 'Import Quantity' in df_pivot.columns else pd.Series([0]*len(df_pivot), index=df_pivot.index), errors='coerce')
    production_qty = pd.to_numeric(df_pivot['Production'] if 'Production' in df_pivot.columns else pd.Series([0]*len(df_pivot), index=df_pivot.index), errors='coerce')
    export_qty = pd.to_numeric(df_pivot['Export Quantity'] if 'Export Quantity' in df_pivot.columns else pd.Series([0]*len(df_pivot), index=df_pivot.index), errors='coerce')

    # Define the df only with net_ammonia [t] as production + import quantities (ignores exports for this analysis)
    df_pivot['net_ammonia [t]'] = production_qty.fillna(0) + import_qty.fillna(0)
    df_pivot['production'] = production_qty.fillna(0)
    df_pivot['import_qty'] = import_qty.fillna(0)
    df_pivot["export_qty"] = export_qty.fillna(0)

    # Reset index to keep all keys as columns for downstream use
    df_net_ammonia = df_pivot[['net_ammonia [t]', 'production', 'import_qty',"export_qty"]].reset_index()

    return df_net_ammonia, df

def producer_countries (df,year):
    # Check which are the most recent producers (has 'Production' element)
    last_year = year # last_year = df['Year'].max()
    df_latest_production = df[df['Year'] == last_year]
    mask = (df['Element'] == 'Production') & (df['Value'].notnull())
    producer_countries = df_latest_production.loc[mask]
    producer_countries.sort_values(by='Value', ascending=False, inplace=True)
    return producer_countries, last_year

def project_ammonia_paths(df, self_sufficiency_paths, cagr, tech_shift, base_year):
    
    if base_year is None:
        base_year = df['year'].max()
    df_proj = df[df['year'] == base_year].copy()

    projection_rows = []

    for proj_year, self_sufficiency_share in self_sufficiency_paths:
        cagr_year = next(val for year, val in cagr if year == proj_year)
        tech_shift_share = next(val for year, val in tech_shift if year == proj_year)
        year_base = 2030 if proj_year == 2050 else base_year

        local_production = pd.to_numeric(df_proj["production"], errors='coerce') * ((1 + cagr_year) ** (proj_year - year_base))
        df_proj["local_production"] =  (local_production * (1 - tech_shift_share)) 

        new_local_production = pd.to_numeric(df_proj["import_qty"], errors='coerce') * ((1 + cagr_year) ** (proj_year - year_base)) * self_sufficiency_share
        df_proj["new_local_production"] = (new_local_production + (local_production * tech_shift_share))
        df_proj["year"] = proj_year

        projection_rows.append(df_proj[["country", "year", "local_production", "new_local_production"]])
        
    # combien all proyections into one df
    df_projection = pd.concat(projection_rows, ignore_index=True)
    df_projection.set_index(['country'], inplace=True)
    return df_projection

def to_MWh(df):
    lhv_nh3 = 5.17  # MWh_NH3/t_NH3
    return df * lhv_nh3

def add_histotical_to_projection(df_projection, df_net_ammonia, year=2019):
    df_yr = df_net_ammonia[df_net_ammonia['year'] == year].copy()
    df_yr = df_yr.set_index('country') if 'country' in df_yr.columns else df_yr
    df_yr = df_yr.drop(columns=['unit'], errors='ignore')
    return pd.concat([df_yr, df_projection], axis=0)


In [120]:
if __name__ == "__main__":
    csv_file_path = r"C:\Users\eduque\Fraunhofer\DeV-KopSys-2 - Globale ESA\04_methods_and_models\Sectoral_End-Use_Energy_Scenario\subsector_workflows\industry\scripts\chemical_and_petrochemical\ammonia\UNdata_Export_20250709_101100979.csv"
    
    # select countries to analyse
    selected_countries = [
        'Chile',
        'Egypt',
        'Morocco',
        'South Africa'
        ]  
    # clean csv data and generate df
    df_net_ammonia, raw_df = clean_ammonia_csv(csv_file_path, selected_countries)

    # display countries that are producing ammonia worldwide
    producer_countries, last_year = producer_countries(raw_df, year = 2019)
    # print(producer_countries)

C:\Users\eduque\AppData\Local\Temp\ipykernel_21828\3860567456.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  producer_countries.sort_values(by='Value', ascending=False, inplace=True)


In [121]:
# Base on expert assumptions
self_sufficiency_paths = {
    "A":[
    (2030, 0.10), # 10% share of imports from 2019
    (2035, 0.30), # 40% share of imports from 2019
    (2050, 0.50), # 85% share of imports from 2019
    ],
    "B": [
    (2030, 0.20), # 20% share of imports from 2019
    (2035, 0.50), # 40% share of imports from 2019
    (2050, 0.85), # 85% share of imports from 2019
    ]}

cagr = {
    "A": 
    [(2030, 0.03), # 3% CAGR from 2019 to 2030
     (2035, 0.03), # 3% CAGR from 2019 to 2035
     (2050, 0.02)], # 2% CAGR from 2019 to 2050,
    "B":
    [(2030, 0.07), # 7% CAGR from 2019 to 2030
     (2035, 0.05), # 5% CAGR from 2019 to 2035
     (2050, 0.03)], # 3% CAGR from 2019 to 2050,
    }

tech_shift_share = {
    "A": 
    [(2030, 0.05),  
     (2035, 0.05), 
     (2050, 0.15)], 
    "B":
    [(2030, 0.10), 
     (2035, 0.12), 
     (2050, 0.30)], 
    }

In [122]:
scenario = "A"
df_scenario_A = project_ammonia_paths(df_net_ammonia, self_sufficiency_paths[scenario], cagr[scenario], tech_shift_share[scenario], base_year = 2019)
df_scenario_A_Mwh = to_MWh(df_scenario_A)

scenario = "B"
df_scenario_B = project_ammonia_paths(df_net_ammonia, self_sufficiency_paths[scenario], cagr[scenario], tech_shift_share[scenario], base_year = 2019)
df_scenario_B_Mwh = to_MWh(df_scenario_B)

In [123]:
df_scenario_A_Mwh

,year,local_production,new_local_production
country,,,
Chile,10495.10,0.000000e+00,2.484784e+05
Egypt,10495.10,3.482243e+07,1.835633e+06
Morocco,10495.10,0.000000e+00,1.135526e+06
South Africa,10495.10,0.000000e+00,5.207872e+04
Chile,10520.95,0.000000e+00,8.641638e+05
Egypt,10520.95,4.036874e+07,2.134666e+06
Morocco,10520.95,0.000000e+00,3.949159e+06
South Africa,10520.95,0.000000e+00,1.811205e+05
Chile,10598.50,0.000000e+00,1.333683e+06


In [124]:
df_scenario_B

,year,local_production,new_local_production
country,,,
Chile,2030,0.000000e+00,1.461639e+05
Egypt,2030,9.702854e+06,1.079786e+06
Morocco,2030,0.000000e+00,6.679574e+05
South Africa,2030,0.000000e+00,3.063457e+04
Chile,2035,0.000000e+00,3.789549e+05
Egypt,2035,9.838908e+06,1.346052e+06
Morocco,2035,0.000000e+00,1.731793e+06
South Africa,2035,0.000000e+00,7.942533e+04
Chile,2050,0.000000e+00,5.330306e+05


In [125]:
df_all_A = add_histotical_to_projection(df_scenario_A, df_net_ammonia)
df_all_A

,year,net_ammonia [t],production,import_qty,export_qty,local_production,new_local_production
country,,,,,,,
Chile,2019,347207.17,0.00,347207.17,334.24,NaN,NaN
Egypt,2019,5125967.23,5121951.22,4016.01,243113.83,NaN,NaN
Morocco,2019,1586708.80,0.00,1586708.80,0.10,NaN,NaN
South Africa,2019,72771.32,0.00,72771.32,9053.34,NaN,NaN
Chile,2030,NaN,NaN,NaN,NaN,0.000000e+00,4.806159e+04
Egypt,2030,NaN,NaN,NaN,NaN,6.735479e+06,3.550548e+05
Morocco,2030,NaN,NaN,NaN,NaN,0.000000e+00,2.196376e+05
South Africa,2030,NaN,NaN,NaN,NaN,0.000000e+00,1.007325e+04
Chile,2035,NaN,NaN,NaN,NaN,0.000000e+00,1.671497e+05
